# 🌊 Ingestão Streaming: eventos de resultado de aluno

Este notebook desenvolve a ingestão streaming da pipeline: um producer simula o sistema de avaliação estadual publicando resultados individuais de alunos do ciclo de 2025 em um tópico Pub/Sub, e um consumer lê esses eventos, valida a estrutura, descarta duplicatas e grava micro-lotes na camada Bronze. Eventos malformados são desviados para uma dead letter queue.

> **Convenção de trabalho:** o desenvolvimento acontece neste notebook, célula a célula, com os resultados salvos. Ao final da etapa, o código estável é promovido para `src/ingestion/prod_02_ingestao_streaming.py`.

**Decisões que governam este notebook** (ver [diário de decisões](../docs/decisoes.md)):
- D-005: o streaming opera no nível do aluno, simulando o ciclo de 2025;
- D-009: a mensageria é o Google Pub/Sub;
- Contrato do evento: [config/schemas/evento_resultado_aluno.md](../config/schemas/evento_resultado_aluno.md).

📚 **Referências das aulas (módulo Fase 2, Data Prepare):**
- *ETL Pipelines, Aula 2 (Integração de dados em tempo real):* padrão publish/subscribe, producer e consumer, semânticas de entrega, event time e processing time, DLQ, idempotência;
- *ETL Pipelines, Aula 3 (Cloud Data):* mapeamento da ingestão streaming na GCP para o Pub/Sub;
- *Notebooks Kafka_Demo Partes 2 a 4:* produção e consumo de eventos, deduplicação no consumidor e dead letter queue, aqui aplicados no serviço gerenciado.

---

**Estratégia de desenvolvimento deste notebook:**

```
criar a infraestrutura -> publicar poucos eventos -> consumir e gravar -> erros e DLQ -> volume
   (tópicos e             (producer, validação        (consumer ->        (malformados,     (carga
    subscriptions)         visual no console)          Bronze)             duplicatas)       realista)
```

O fluxo completo é validado com poucos eventos antes de qualquer volume, seguindo o mesmo princípio da ingestão batch: erro barato primeiro.

> ⚠️ **Este notebook não deve ser executado com Run All.** Ele é um registro interativo de desenvolvimento: a Seção 5 reexecuta deliberadamente células anteriores (o consumer da Seção 4), e uma execução linear não reproduziria a narrativa nem os resultados. A execução de ponta a ponta, com um único comando, é papel do script de produção `src/ingestion/prod_02_ingestao_streaming.py`.


## 1. Setup do ambiente

**Passos desta seção:** ler a configuração do projeto (`config/config.json`, o mesmo arquivo usado pela produção), autorizar o acesso e definir os nomes dos recursos de mensageria.

🎓 **Conceito** (*ETL Pipelines, Aula 2*): a infraestrutura de streaming tem quatro peças nomeadas: o **tópico de eventos** (onde o producer publica), a **subscription da pipeline** (por onde o consumer lê, com confirmação individual de cada mensagem), o **tópico de DLQ** (destino dos eventos malformados) e a **subscription de inspeção da DLQ** (para auditar o que foi rejeitado).

> 📌 **Nota para reprodução:** este notebook lê o `config/config.json`, que não é versionado. Crie o seu a partir do `config/config.example.json`, como descrito na seção Como Executar do README.

In [3]:
# --- 1.1 Configuração, credencial e nomes dos recursos ---
import json
from pathlib import Path
from datetime import datetime, timezone

import pydata_google_auth
from google.cloud import pubsub_v1

# Mesma configuração da produção: projeto e bucket de quem executa
CFG = json.loads(Path("../config/config.json").read_text(encoding="utf-8"))
PROJETO_GCP = CFG["projeto_gcp"]
BUCKET_LAKE = CFG["bucket_lake"]

# Credencial ampla (BigQuery + Storage + Pub/Sub), já autorizada nas etapas anteriores
ESCOPOS = ["https://www.googleapis.com/auth/cloud-platform"]
credenciais = pydata_google_auth.get_user_credentials(ESCOPOS)
credenciais = credenciais.with_quota_project(PROJETO_GCP)

# Nomes dos recursos de mensageria (contrato: evento_resultado_aluno v1)
TOPICO_EVENTOS = "eventos-resultado-aluno"
TOPICO_DLQ     = "eventos-resultado-aluno-dlq"
SUB_PIPELINE   = "sub-pipeline-bronze"      # o consumer da pipeline lê por aqui
SUB_DLQ        = "sub-inspecao-dlq"         # janela de auditoria da DLQ

print(f"Projeto: {PROJETO_GCP}")
print(f"Tópico de eventos: {TOPICO_EVENTOS}")
print(f"Tópico de DLQ:     {TOPICO_DLQ}")

Projeto: tech-challenge-fase2
Tópico de eventos: eventos-resultado-aluno
Tópico de DLQ:     eventos-resultado-aluno-dlq


## 2. Criação da infraestrutura de mensageria

**Passos desta seção:** criar os dois tópicos e as duas subscriptions, de forma idempotente, e listar o que existe no projeto.

🎓 **Conceito, tópico e subscription** (*ETL Pipelines, Aulas 2 e 3*): no Pub/Sub, o tópico é o mural onde os eventos são publicados. A subscription é o vínculo de um leitor com o mural: ela guarda a posição de leitura e recebe a cópia de cada evento. Cada mensagem lida precisa ser confirmada (**ack**); mensagens não confirmadas são reentregues. É o equivalente gerenciado do consumer group com offsets das aulas de Kafka.

🎓 **Conceito, DLQ em nível de aplicação** (*notebook Kafka_Demo Parte 4*): a DLQ deste projeto é um segundo tópico, para onde o próprio consumer publica os eventos que reprovam na validação estrutural. O Pub/Sub também oferece dead letter policy nativa (reencaminhamento automático após N tentativas de entrega), registrada como evolução possível para produção.

ℹ️ **Observação:** a criação é idempotente: recursos que já existem são reaproveitados, sem erro. O mesmo princípio das partições da Bronze.

In [4]:
# --- 2.1 Criar tópicos e subscriptions (idempotente) ---
from google.api_core.exceptions import AlreadyExists

publisher  = pubsub_v1.PublisherClient(credentials=credenciais)
subscriber = pubsub_v1.SubscriberClient(credentials=credenciais)

def criar_topico(nome: str) -> str:
    caminho = publisher.topic_path(PROJETO_GCP, nome)
    try:
        publisher.create_topic(request={"name": caminho})
        print(f"Tópico criado:        {nome}")
    except AlreadyExists:
        print(f"Tópico já existia:    {nome}")
    return caminho

def criar_subscription(nome: str, topico: str) -> str:
    caminho = subscriber.subscription_path(PROJETO_GCP, nome)
    try:
        subscriber.create_subscription(request={"name": caminho, "topic": topico})
        print(f"Subscription criada:  {nome} -> {topico.split('/')[-1]}")
    except AlreadyExists:
        print(f"Subscription existia: {nome}")
    return caminho

caminho_topico = criar_topico(TOPICO_EVENTOS)
caminho_dlq    = criar_topico(TOPICO_DLQ)
caminho_sub    = criar_subscription(SUB_PIPELINE, caminho_topico)
caminho_subdlq = criar_subscription(SUB_DLQ, caminho_dlq)

print()
print("Infraestrutura de mensageria pronta.")

Tópico criado:        eventos-resultado-aluno
Tópico criado:        eventos-resultado-aluno-dlq
Subscription criada:  sub-pipeline-bronze -> eventos-resultado-aluno
Subscription criada:  sub-inspecao-dlq -> eventos-resultado-aluno-dlq

Infraestrutura de mensageria pronta.


## 3. Producer: publicando resultados do ciclo de 2025

**Passos desta seção:** (3.1) recriar a subscription da pipeline com entrega ordenada e carregar a base de amostragem da Bronze; (3.2) definir a função que gera eventos conforme o contrato e publicar um primeiro lote pequeno no tópico.

🎓 **Conceito, o papel do producer** (*ETL Pipelines, Aula 2*): o producer representa o sistema externo (a avaliação estadual). Ele publica o evento no tópico e encerra sua responsabilidade: não conhece o consumer, a Bronze nem o destino dos dados. Na simulação, ele lê os microdados reais da Bronze (ciclo 2024) e emite resultados plausíveis para o ciclo de 2025, preservando as distribuições de proficiência e peso da fonte, conforme a regra 4 do [contrato](../config/schemas/evento_resultado_aluno.md).

🎓 **Conceito, entrega ordenada** (*ETL Pipelines, Aula 2; equivalente à partição com chave do Kafka*): o contrato define `id_municipio` como ordering key: eventos do mesmo município são entregues na ordem de publicação. No Pub/Sub, a ordenação é definida na criação da subscription; como a nossa foi criada sem essa opção (e nenhuma mensagem trafegou ainda), ela é recriada aqui com `enable_message_ordering`.

ℹ️ **Observação:** o primeiro lote é proposital e pequeno (10 eventos), seguindo o mesmo princípio da ingestão batch: validar o fluxo completo com custo de erro mínimo antes de qualquer volume.

> ⚠️ **Sobre a execução das células que publicam eventos (esta e as demais):** após imprimir as confirmações (`message_id`), a célula pode permanecer com o indicador de execução ativo indefinidamente. O trabalho já terminou: cada `message_id` é o recibo do broker de que o evento foi recebido e retido no tópico. O que mantém o indicador ativo é a infraestrutura de fundo do cliente Pub/Sub (threads de envio em lote e canais de rede), que em um script se encerra com o processo, mas em um notebook permanece viva no kernel. Se ocorrer, é seguro interromper: clique no quadrado de parada (⏹) ao lado da célula ou use `Ctrl+Shift+P` seguido de "Jupyter: Interrupt Kernel". Os eventos publicados não são afetados. Na versão de produção (`src/`), o encerramento dos clientes é feito de forma explícita.


In [5]:
# --- 3.1 Entrega ordenada + base de amostragem da Bronze ---
import pandas as pd
from google.api_core.exceptions import NotFound
from google.cloud import storage

# Recriar a subscription da pipeline com entrega ordenada (exigência do
# contrato). Seguro aqui: nenhuma mensagem foi publicada ainda.
try:
    subscriber.delete_subscription(request={"subscription": caminho_sub})
except NotFound:
    pass
subscriber.create_subscription(
    request={"name": caminho_sub, "topic": caminho_topico,
             "enable_message_ordering": True}
)
print(f"Subscription recriada com entrega ordenada: {SUB_PIPELINE}")

# Descobrir a particao mais recente de alunos na Bronze (sem data fixa no codigo)
cliente_storage = storage.Client(project=PROJETO_GCP, credentials=credenciais)
particoes = sorted({
    b.name.split("/")[2]
    for b in cliente_storage.list_blobs(BUCKET_LAKE, prefix="bronze/alunos/")
})
ultima = particoes[-1]
caminho_alunos = f"gs://{BUCKET_LAKE}/bronze/alunos/{ultima}/alunos.parquet"
print(f"Particao mais recente da Bronze: {ultima}")

# Base de amostragem: alunos presentes de 2024 (proficiencia e peso validos),
# apenas as colunas que o contrato precisa
base = pd.read_parquet(
    caminho_alunos,
    columns=["ano", "id_municipio", "rede", "proficiencia", "peso_aluno"],
    storage_options={"token": credenciais},
)
base = base[(base["ano"] == 2024)
            & base["proficiencia"].notna()
            & base["peso_aluno"].notna()]
print(f"Base de amostragem carregada: {len(base):,} alunos presentes de 2024")

Subscription recriada com entrega ordenada: sub-pipeline-bronze
Particao mais recente da Bronze: data_ingestao=2026-07-09
Base de amostragem carregada: 1,851,852 alunos presentes de 2024


In [7]:
# --- 3.2 Gerar eventos conforme o contrato e publicar o primeiro lote ---
import random
import uuid

def gerar_evento(linha) -> dict:
    """Monta um evento v1 (contrato evento_resultado_aluno) a partir de um
    aluno real da base de amostragem."""
    return {
        "id_evento":      str(uuid.uuid4()),
        "schema_version": 1,
        "event_time":     datetime.now(timezone.utc).isoformat(),
        "id_municipio":   str(linha["id_municipio"]),
        "rede":           str(linha["rede"]),
        "id_aluno":       f"A2025-{random.randint(0, 999999):06d}",
        "proficiencia":   round(float(linha["proficiencia"]), 1),
        "peso_aluno":     round(float(linha["peso_aluno"]), 4),
    }

# Publisher com ordenacao habilitada (par da subscription ordenada)
publisher_ord = pubsub_v1.PublisherClient(
    credentials=credenciais,
    publisher_options=pubsub_v1.types.PublisherOptions(enable_message_ordering=True),
)

N_EVENTOS = 10
amostra = base.sample(N_EVENTOS, random_state=42)

print(f"Publicando {N_EVENTOS} eventos no tópico {TOPICO_EVENTOS}...")
print()
for _, linha in amostra.iterrows():
    evento = gerar_evento(linha)
    dados = json.dumps(evento).encode("utf-8")
    futuro = publisher_ord.publish(
        caminho_topico, dados, ordering_key=evento["id_municipio"]
    )
    message_id = futuro.result(timeout=30)   # aguarda a confirmacao do broker
    print(f"  municipio {evento['id_municipio']}  prof {evento['proficiencia']:>6}"
          f"  -> message_id {message_id}")

print()
print("Exemplo de evento publicado (o payload que viaja no topico):")
print(json.dumps(evento, indent=2, ensure_ascii=False))

Publicando 10 eventos no tópico eventos-resultado-aluno...

  municipio 3136702  prof  738.6  -> message_id 20556241711931074
  municipio 1501303  prof  673.1  -> message_id 20556304516099325
  municipio 2615300  prof  730.4  -> message_id 20556369394297374
  municipio 3305554  prof  661.9  -> message_id 20556333211917140
  municipio 3106200  prof  744.3  -> message_id 19867467560766289
  municipio 5103403  prof  737.6  -> message_id 19864149897564440
  municipio 3535606  prof  798.5  -> message_id 20556282016518684
  municipio 5208707  prof  784.1  -> message_id 20556178065597052
  municipio 3201506  prof  807.0  -> message_id 20556118692682523
  municipio 3552809  prof  694.3  -> message_id 20556313703566082

Exemplo de evento publicado (o payload que viaja no topico):
{
  "id_evento": "8988fa42-f9c5-4885-a5a1-996dfab0f8d3",
  "schema_version": 1,
  "event_time": "2026-07-10T19:23:16.891305+00:00",
  "id_municipio": "3552809",
  "rede": "3",
  "id_aluno": "A2025-734794",
  "proficien

## 4. Consumer: do backlog ao primeiro micro-lote na Bronze

**Passos desta seção:** (4.1) ler o backlog da subscription, validar cada evento contra o contrato, desviar malformados para a DLQ, descartar duplicatas e confirmar (ack) as mensagens processadas; (4.2) gravar os eventos válidos como micro-lote Parquet na área de streaming da Bronze e conferir o resultado.

🎓 **Conceito, pull e ack** (*ETL Pipelines, Aula 2; equivalente ao offset commit do Kafka*): o consumer pede mensagens à subscription (pull) e, após processar cada uma, confirma o recebimento (ack). Mensagens não confirmadas voltam ao backlog e são reentregues, o que caracteriza a semântica at-least-once: nada se perde, mas duplicatas são possíveis, e por isso o consumer é idempotente.

🎓 **Conceito, deduplicação** (*notebook Kafka_Demo Parte 4, Etapa 5*): o `id_evento` do contrato é a chave de idempotência. O consumer mantém o registro dos ids já processados e descarta repetições. Nesta fase de desenvolvimento o registro vive em memória; na promoção para produção ele será persistido.

🎓 **Conceito, a Bronze do streaming cresce por acúmulo:** diferente das tabelas batch (sobrescrita da partição do dia), cada execução do consumer grava um **novo** arquivo de micro-lote. A idempotência aqui não vem da sobrescrita, e sim da deduplicação por `id_evento`. Cada registro carrega o `event_time` (do payload) e o `_processing_timestamp` (da gravação), os dois tempos da Aula 2.

In [12]:
# --- 4.1 Consumir o backlog: validar, desviar, deduplicar, confirmar ---
CAMPOS_OBRIGATORIOS = {
    "id_evento", "schema_version", "event_time", "id_municipio",
    "rede", "id_aluno", "proficiencia", "peso_aluno",
}

def validar_evento(ev) -> str:
    """Valida um evento contra o contrato v1. Devolve o motivo da falha
    ou uma string vazia se o evento for valido."""
    if not isinstance(ev, dict):
        return "payload nao e um objeto JSON"
    faltantes = CAMPOS_OBRIGATORIOS - ev.keys()
    if faltantes:
        return f"campos ausentes: {sorted(faltantes)}"
    if ev["schema_version"] != 1:
        return f"schema_version desconhecida: {ev['schema_version']}"
    if not isinstance(ev["proficiencia"], (int, float)):
        return "proficiencia nao numerica"
    if not isinstance(ev["peso_aluno"], (int, float)):
        return "peso_aluno nao numerico"
    return ""

publisher_dlq   = pubsub_v1.PublisherClient(credentials=credenciais)
ids_processados = set()     # estado de deduplicacao (em memoria nesta fase)
validos         = []
contagem        = {"lidos": 0, "validos": 0, "duplicados": 0, "malformados": 0}

# Pull sincrono em rodadas, ate o backlog esvaziar.
# Quando nao ha mensagens disponiveis dentro do prazo, o pull lanca
# DeadlineExceeded (504) em vez de devolver resposta vazia; a excecao
# e tratada como fim do backlog (padrao recomendado para pull sincrono).
from google.api_core.exceptions import DeadlineExceeded

while True:
    try:
        resposta = subscriber.pull(
            request={"subscription": caminho_sub, "max_messages": 50},
            timeout=15,
        )
    except DeadlineExceeded:
        break
    if not resposta.received_messages:
        break

    ack_ids = []
    for recebida in resposta.received_messages:
        contagem["lidos"] += 1
        try:
            evento = json.loads(recebida.message.data.decode("utf-8"))
        except json.JSONDecodeError:
            evento = None

        motivo = validar_evento(evento) if evento is not None else "JSON invalido"
        if motivo:
            # Malformado: desvia para a DLQ com o motivo, sem travar o fluxo
            publisher_dlq.publish(
                caminho_dlq, recebida.message.data, motivo=motivo
            ).result(timeout=30)
            contagem["malformados"] += 1
        elif evento["id_evento"] in ids_processados:
            contagem["duplicados"] += 1
        else:
            ids_processados.add(evento["id_evento"])
            validos.append(evento)
            contagem["validos"] += 1

        ack_ids.append(recebida.ack_id)   # processada (ou desviada): confirmar

    subscriber.acknowledge(
        request={"subscription": caminho_sub, "ack_ids": ack_ids}
    )

print("Backlog consumido:")
for chave, valor in contagem.items():
    print(f"  {chave:<12} {valor}")

Backlog consumido:
  lidos        7
  validos      2
  duplicados   1
  malformados  4


In [14]:
# --- 4.2 Gravar o micro-lote na Bronze e conferir ---
import pandas as pd

if not validos:
    print("Nenhum evento valido para gravar (backlog vazio?).")
else:
    agora = datetime.now(timezone.utc)
    df_lote = pd.DataFrame(validos)

    # Os dois tempos do streaming: o do payload (event_time, ja presente)
    # e o do processamento (registrado agora, na gravacao)
    df_lote["_processing_timestamp"] = agora.isoformat()
    df_lote["_source"] = f"pubsub:{TOPICO_EVENTOS}"

    # Micro-lote: arquivo NOVO por execucao (acumulo), nomeado pelo instante
    dia  = agora.strftime("%Y-%m-%d")
    lote = agora.strftime("%H%M%S")
    caminho_lote = (
        f"gs://{BUCKET_LAKE}/bronze/eventos_resultado_aluno/"
        f"data_processamento={dia}/lote_{lote}.parquet"
    )
    df_lote.to_parquet(caminho_lote, index=False,
                       storage_options={"token": credenciais})
    print(f"Micro-lote gravado: {caminho_lote}")
    print(f"Eventos no lote:    {len(df_lote)}")
    print()

    # Conferencia: ler de volta do lake (round-trip, como na ingestao batch)
    df_conferencia = pd.read_parquet(caminho_lote,
                                     storage_options={"token": credenciais})
    print(f"Conferencia (lido de volta do lake): {len(df_conferencia)} eventos")
    df_conferencia[["id_evento", "id_municipio", "proficiencia",
                    "peso_aluno", "event_time", "_processing_timestamp"]].head()

Micro-lote gravado: gs://tech-challenge-fase2-lake-rm373453/bronze/eventos_resultado_aluno/data_processamento=2026-07-10/lote_194340.parquet
Eventos no lote:    2

Conferencia (lido de volta do lake): 2 eventos


## 5. Teste de resiliência: eventos defeituosos e duplicata verdadeira

**Passos desta seção:** (5.1) publicar um lote de sabotagem com eventos válidos, malformados de quatro tipos e uma duplicata verdadeira de `id_evento`; em seguida, **reexecutar as células 4.1 e 4.2** (o consumer é reutilizável); (5.2) inspecionar a DLQ e conferir os rejeitados com o motivo de cada um.

🎓 **Conceito, sabotagem controlada** (*notebook Kafka_Demo Parte 4, Etapa 4*): a prova de que uma pipeline de streaming é resiliente não é processar o caminho feliz, e sim receber dado ruim sem travar: o evento defeituoso vai para a DLQ com o motivo, o fluxo segue, e a auditoria acontece depois, na janela de inspeção.

🎓 **Conceito, duplicata verdadeira** (*notebook Kafka_Demo Parte 4, Etapa 5*): na Seção 4, os 20 eventos tinham UUIDs distintos e nada foi deduplicado, corretamente. Aqui o mesmo `id_evento` é publicado duas vezes, simulando a reentrega do at-least-once, e o consumer deve descartar a segunda ocorrência.

ℹ️ **Observação:** o registro de deduplicação (`ids_processados`) vive na memória do notebook e é reiniciado a cada execução da 4.1; por isso a duplicata do teste está dentro do mesmo lote. Na promoção para produção, o registro será persistido entre execuções.

**Roteiro de execução desta seção** (o consumer das células 4.1 e 4.2 é reutilizado):

1. Execute a célula **5.1** (publica a sabotagem);
2. Volte e **reexecute a célula 4.1** (o consumer processa o lote defeituoso);
3. **Reexecute a célula 4.2** (grava o micro-lote com os válidos);
4. Siga para a célula **5.2** (inspeção da DLQ com os motivos das rejeições).


In [11]:
# --- 5.1 Publicar o lote de sabotagem ---
# Composicao: 2 validos + 1 duplicata verdadeira + 4 malformados
evento_a = gerar_evento(base.sample(1, random_state=7).iloc[0])
evento_b = gerar_evento(base.sample(1, random_state=8).iloc[0])

malformado_sem_campo = {k: v for k, v in gerar_evento(base.sample(1).iloc[0]).items()
                        if k != "peso_aluno"}                       # campo obrigatorio ausente
malformado_tipo      = {**gerar_evento(base.sample(1).iloc[0]),
                        "proficiencia": "setecentos e cinquentaa e dois mané"}   # tipo invalido
malformado_versao    = {**gerar_evento(base.sample(1).iloc[0]),
                        "schema_version": 99}                       # versao desconhecida

lote_sabotagem = [
    ("valido A",            evento_a),
    ("valido B",            evento_b),
    ("duplicata de A",      evento_a),          # mesmo id_evento publicado 2x
    ("sem peso_aluno",      malformado_sem_campo),
    ("proficiencia texto",  malformado_tipo),
    ("schema_version 99",   malformado_versao),
]

for rotulo, ev in lote_sabotagem:
    dados = json.dumps(ev).encode("utf-8")
    publisher_ord.publish(caminho_topico, dados,
                          ordering_key=ev.get("id_municipio", "sabotagem")
                          ).result(timeout=30)
    print(f"  publicado: {rotulo}")

# O quarto malformado nem JSON e: bytes crus quebrados
publisher_ord.publish(caminho_topico, b"{isso nao e um json valido",
                      ordering_key="sabotagem").result(timeout=30)
print("  publicado: JSON quebrado (bytes crus)")

print()
print("Lote de sabotagem no topico: 7 mensagens (2 validas, 1 duplicata, 4 malformadas)")
print()
print(">>> Agora REEXECUTE a celula 4.1 (consumer) e depois a 4.2 (gravacao).")
print(">>> Resultado esperado na 4.1: lidos 7, validos 2, duplicados 1, malformados 4")
print(">>> Em seguida, siga para a celula 5.2 (inspecao da DLQ).")

  publicado: valido A
  publicado: valido B
  publicado: duplicata de A
  publicado: sem peso_aluno
  publicado: proficiencia texto
  publicado: schema_version 99
  publicado: JSON quebrado (bytes crus)

Lote de sabotagem no topico: 7 mensagens (2 validas, 1 duplicata, 4 malformadas)

>>> Agora REEXECUTE a celula 4.1 (consumer) e depois a 4.2 (gravacao).
>>> Resultado esperado na 4.1: lidos 7, validos 2, duplicados 1, malformados 4


In [15]:
# --- 5.2 Inspecionar a DLQ: os rejeitados e seus motivos ---
# A janela de auditoria: le a subscription da DLQ e mostra motivo + payload.
# Atencao: a inspecao confirma (ack) as mensagens; em producao, a DLQ teria
# um fluxo proprio de tratamento ou arquivamento.
from google.api_core.exceptions import DeadlineExceeded

rejeitados = []
while True:
    try:
        resp = subscriber.pull(
            request={"subscription": caminho_subdlq, "max_messages": 20},
            timeout=15,
        )
    except DeadlineExceeded:
        break
    if not resp.received_messages:
        break
    ack_ids = []
    for m in resp.received_messages:
        rejeitados.append({
            "motivo":  m.message.attributes.get("motivo", "(sem motivo)"),
            "payload": m.message.data.decode("utf-8", errors="replace")[:70],
        })
        ack_ids.append(m.ack_id)
    subscriber.acknowledge(request={"subscription": caminho_subdlq,
                                    "ack_ids": ack_ids})

print(f"Mensagens na DLQ: {len(rejeitados)}")
print()
for r in rejeitados:
    print(f"  motivo: {r['motivo']}")
    print(f"  payload: {r['payload']}...")
    print()

Mensagens na DLQ: 4

  motivo: JSON invalido
  payload: {isso nao e um json valido...

  motivo: schema_version desconhecida: 99
  payload: {"id_evento": "67d06573-e473-420b-8c73-1afad3a7188f", "schema_version"...

  motivo: campos ausentes: ['peso_aluno']
  payload: {"id_evento": "c7f40b3b-f9d8-44b7-9ac3-90341b7750ae", "schema_version"...

  motivo: proficiencia nao numerica
  payload: {"id_evento": "8cd57928-3eb2-4a8a-95b9-d2d99f112a9a", "schema_version"...



## 6. Volume e generalização: o ensaio da produção

**Passos desta seção:** (6.1) empacotar a publicação em uma função e publicar um lote de 2.000 eventos, medindo a vazão; (6.2) empacotar o consumo em uma função, consumir todo o backlog com medição, gravar o micro-lote e apresentar o inventário final da área de streaming da Bronze.

🎓 **Conceito, vazão (throughput)** (*ETL Pipelines, Aula 2*): eventos por segundo é a métrica de capacidade do fluxo, par do lag no monitoramento de streaming. Medi-la aqui estabelece a linha de base que a Etapa de monitoramento usará.

🎓 **Conceito, ensaio da produção** (mesmo princípio da ingestão batch): o ciclo validado nas Seções 3 a 5 é empacotado nas funções `publicar_lote()` e `consumir_e_gravar()`, as unidades que serão promovidas para `src/ingestion/prod_02_ingestao_streaming.py`.

ℹ️ **Observação:** o consumo usa pull síncrono, adequado ao ambiente didático e a micro-lotes. Em produção com fluxo contínuo, a alternativa de maior vazão é a assinatura assíncrona (streaming pull), registrada como evolução possível.

In [16]:
# --- 6.1 Publicação em volume, com medição de vazão ---
import time

def publicar_lote(n: int) -> float:
    """Publica n eventos amostrados da base e devolve a duracao em segundos.

    Coleta os futuros e aguarda todos ao final: mais rapido que aguardar
    um a um, pois o cliente envia em lotes pela infraestrutura de fundo.
    """
    amostra_lote = base.sample(n)
    futuros = []
    for _, linha in amostra_lote.iterrows():
        ev = gerar_evento(linha)
        futuros.append(publisher_ord.publish(
            caminho_topico,
            json.dumps(ev).encode("utf-8"),
            ordering_key=ev["id_municipio"],
        ))
    for f in futuros:
        f.result(timeout=60)      # aguarda a confirmacao de todos
    return time.time() - t0

N_VOLUME = 2000
t0 = time.time()
duracao_pub = publicar_lote(N_VOLUME)

print(f"Publicados {N_VOLUME:,} eventos em {duracao_pub:.1f}s "
      f"({N_VOLUME/duracao_pub:,.0f} eventos/s)")

Publicados 2,000 eventos em 4.3s (469 eventos/s)


In [17]:
# --- 6.2 Consumo em volume, gravacao e inventario final ---
from google.api_core.exceptions import DeadlineExceeded

def consumir_e_gravar() -> dict:
    """Consome todo o backlog (validando, desviando e deduplicando),
    grava o micro-lote na Bronze e devolve as metricas da execucao."""
    ids_vistos = set()
    eventos_validos = []
    metricas = {"lidos": 0, "validos": 0, "duplicados": 0, "malformados": 0}
    t0 = time.time()

    while True:
        try:
            resposta = subscriber.pull(
                request={"subscription": caminho_sub, "max_messages": 500},
                timeout=15,
            )
        except DeadlineExceeded:
            break
        if not resposta.received_messages:
            break

        ack_ids = []
        for recebida in resposta.received_messages:
            metricas["lidos"] += 1
            try:
                ev = json.loads(recebida.message.data.decode("utf-8"))
            except json.JSONDecodeError:
                ev = None
            motivo = validar_evento(ev) if ev is not None else "JSON invalido"
            if motivo:
                publisher_dlq.publish(caminho_dlq, recebida.message.data,
                                      motivo=motivo)
                metricas["malformados"] += 1
            elif ev["id_evento"] in ids_vistos:
                metricas["duplicados"] += 1
            else:
                ids_vistos.add(ev["id_evento"])
                eventos_validos.append(ev)
                metricas["validos"] += 1
            ack_ids.append(recebida.ack_id)
        subscriber.acknowledge(request={"subscription": caminho_sub,
                                        "ack_ids": ack_ids})

    metricas["duracao_s"] = time.time() - t0

    # Gravacao do micro-lote (mesmo padrao da Secao 4.2)
    if eventos_validos:
        agora = datetime.now(timezone.utc)
        df = pd.DataFrame(eventos_validos)
        df["_processing_timestamp"] = agora.isoformat()
        df["_source"] = f"pubsub:{TOPICO_EVENTOS}"
        destino = (f"gs://{BUCKET_LAKE}/bronze/eventos_resultado_aluno/"
                   f"data_processamento={agora:%Y-%m-%d}/lote_{agora:%H%M%S}.parquet")
        df.to_parquet(destino, index=False, storage_options={"token": credenciais})
        metricas["arquivo"] = destino
    return metricas

resultado = consumir_e_gravar()

print("Consumo em volume:")
for chave in ("lidos", "validos", "duplicados", "malformados"):
    print(f"  {chave:<12} {resultado[chave]:,}")
print(f"  duracao      {resultado['duracao_s']:.1f}s "
      f"({resultado['lidos']/resultado['duracao_s']:,.0f} eventos/s)")
print(f"  arquivo      {resultado.get('arquivo', '(nada gravado)')}")
print()

# Inventario final da area de streaming da Bronze
print("Inventario de bronze/eventos_resultado_aluno/:")
total_bytes = 0
for b in cliente_storage.list_blobs(BUCKET_LAKE, prefix="bronze/eventos_resultado_aluno/"):
    total_bytes += b.size
    print(f"  {b.name}  ({b.size/1024:.1f} KB)")
print(f"  total: {total_bytes/1024:.1f} KB")

Consumo em volume:
  lidos        2,000
  validos      2,000
  duplicados   0
  malformados  0
  duracao      22.9s (87 eventos/s)
  arquivo      gs://tech-challenge-fase2-lake-rm373453/bronze/eventos_resultado_aluno/data_processamento=2026-07-10/lote_195505.parquet

Inventario de bronze/eventos_resultado_aluno/:
  bronze/eventos_resultado_aluno/data_processamento=2026-07-10/lote_193611.parquet  (8.0 KB)
  bronze/eventos_resultado_aluno/data_processamento=2026-07-10/lote_194340.parquet  (6.6 KB)
  bronze/eventos_resultado_aluno/data_processamento=2026-07-10/lote_195505.parquet  (132.9 KB)
  total: 147.5 KB


## 7. Consulta de validação: lendo o fluxo consolidado da Bronze

**Passos desta seção:** ler todos os micro-lotes acumulados de uma só vez, apontando para a pasta da área de streaming, e calcular a prévia da taxa ponderada de alfabetização do fluxo, a mesma conta que a camada Silver fará oficialmente.

🎓 **Conceito, leitura consolidada:** apontar a leitura para a pasta (`bronze/eventos_resultado_aluno/`) em vez de um arquivo faz os micro-lotes se unirem automaticamente em uma única base, com a partição `data_processamento` exposta como coluna. O acúmulo do streaming vira base consultável, sem passo intermediário.

🎓 **Conceito, prévia da estimativa** (decisão D-005): a taxa ponderada calculada aqui é apenas uma prévia sobre o fluxo bruto; a versão oficial da estimativa nasce na Silver, com as regras de qualidade aplicadas, identificada como estimativa da pipeline e distinta dos valores oficiais consolidados.

> ⚠️ **Sobre credenciais em sessões longas:** os tokens de acesso da credencial expiram em cerca de uma hora. Se uma leitura ou gravação no lake falhar com `Invalid Credentials, 401` após muito tempo de sessão, a causa é o token vencido retido no cache do `gcsfs`. A célula 7.1 renova a credencial e limpa o cache preventivamente; o mesmo tratamento vale para qualquer célula que acesse o `gs://` após longa inatividade. Nos scripts de produção o problema não ocorre: processos curtos nascem, obtêm token novo e terminam antes da expiração.


In [19]:
# --- 7.1 Ler o fluxo consolidado (todos os micro-lotes de uma vez) ---
# Renovação preventiva da credencial: tokens de acesso expiram em ~1 hora.
# Em sessões longas de notebook, o gcsfs pode reter em cache uma instância
# com token vencido (erro 401); renovar e limpar o cache resolve.
import google.auth.transport.requests
import gcsfs

credenciais.refresh(google.auth.transport.requests.Request())
gcsfs.GCSFileSystem.clear_instance_cache()

caminho_stream = f"gs://{BUCKET_LAKE}/bronze/eventos_resultado_aluno/"
df_stream = pd.read_parquet(caminho_stream, storage_options={"token": credenciais})

print(f"Eventos acumulados no fluxo: {len(df_stream):,}")
print(f"Municípios distintos:        {df_stream['id_municipio'].nunique():,}")
print(f"Lotes (micro-arquivos):      {df_stream['_processing_timestamp'].nunique()}")
print()

# Prévia da estimativa 2025: taxa ponderada de alfabetização do fluxo
# (a conta oficial da Silver: soma dos pesos dos alfabetizados / soma dos pesos)
alfabetizados = df_stream["proficiencia"] >= 743
taxa = (df_stream["peso_aluno"] * alfabetizados).sum() / df_stream["peso_aluno"].sum() * 100
print(f"Prévia da estimativa do fluxo (taxa ponderada): {taxa:.1f}%")

df_stream.head(5)

Eventos acumulados no fluxo: 2,022
Municípios distintos:        1,084
Lotes (micro-arquivos):      3

Prévia da estimativa do fluxo (taxa ponderada): 59.9%


,id_evento,schema_version,event_time,id_municipio,rede,id_aluno,proficiencia,peso_aluno,_processing_timestamp,_source,data_processamento
0,e5f5687a-c03e-476a-b261-4025db412967,1,2026-07-10T19:10:18.840558+00:00,3201506,3,A2025-583304,807.0,1.1100,2026-07-10T19:36:11.652160+00:00,pubsub:eventos-resultado-aluno,2026-07-10
1,b77f6d64-148a-41b1-bb32-94504619315b,1,2026-07-10T19:23:16.649907+00:00,3201506,3,A2025-612760,807.0,1.1100,2026-07-10T19:36:11.652160+00:00,pubsub:eventos-resultado-aluno,2026-07-10
2,edc6b632-8005-4d5a-bec3-e7c357aeb032,1,2026-07-10T19:10:19.124770+00:00,3552809,3,A2025-283077,694.3,1.2353,2026-07-10T19:36:11.652160+00:00,pubsub:eventos-resultado-aluno,2026-07-10
3,8988fa42-f9c5-4885-a5a1-996dfab0f8d3,1,2026-07-10T19:23:16.891305+00:00,3552809,3,A2025-734794,694.3,1.2353,2026-07-10T19:36:11.652160+00:00,pubsub:eventos-resultado-aluno,2026-07-10
4,58aae0f8-7094-4795-ba94-8536750efd74,1,2026-07-10T19:10:16.613873+00:00,3136702,3,A2025-580297,738.6,1.4400,2026-07-10T19:36:11.652160+00:00,pubsub:eventos-resultado-aluno,2026-07-10
